# Notebook 07 — 3D Conformers: From 2D Graphs to 3D Molecular Shapes

Drug-receptor interactions happen in three dimensions. A 2D molecular graph encodes **connectivity** — which atoms are bonded to which — but tells you nothing about **shape**. The same connectivity can fold into many different 3D arrangements (conformations), and only certain shapes will fit into a binding pocket.

Conformer generation bridges this gap: starting from a SMILES string (pure topology), we produce realistic 3D coordinate sets that capture the spatial reality of how a molecule actually exists in solution or at a receptor.

In this notebook we will:
1. Generate a single 3D conformer using distance geometry
2. Optimise geometry with molecular mechanics force fields (MMFF94, UFF)
3. Sample conformational ensembles
4. Analyse conformer geometry (distances, RMSD matrices)
5. Align molecules for pharmacophore comparison
6. Write 3D structures to file

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolAlign, rdMolDescriptors
from rdkit.Chem import TorsionFingerprints
import numpy as np

## Why 3D Matters

**Wet-lab refresher:** Molecular recognition is inherently three-dimensional. A binding pocket has a specific shape, charge distribution, and set of hydrogen-bond donors/acceptors arranged in space. A molecule must adopt a complementary conformation to bind effectively.

The most vivid example: **enantiomers** share the exact same 2D graph and the same molecular formula, yet they can have completely different biological activities. (R)-thalidomide is a sedative; (S)-thalidomide is teratogenic. Same connectivity, different 3D arrangement, dramatically different pharmacology.

Other situations where 3D is critical:
- **Conformational gating** — some kinase inhibitors only bind when the protein's activation loop is in a specific conformation (DFG-in vs. DFG-out).
- **NMR observables** — NOE distances, J-couplings, and RDCs are all functions of 3D geometry averaged over the conformational ensemble.
- **Docking and virtual screening** — you need 3D coordinates to score poses inside a protein structure.

Bottom line: if you are doing anything beyond simple property prediction, you need 3D.

## Single Conformer Generation

**Computational concept — Distance Geometry:**

If you have ever assembled a Dreiding model (those ball-and-stick kits in the teaching lab), you already understand distance geometry intuitively. You know that a C-C single bond is ~1.54 A, a C=C double bond is ~1.34 A, and tetrahedral angles are ~109.5 degrees. Given these constraints, you can build a physically reasonable 3D structure.

RDKit's `EmbedMolecule` does the same thing algorithmically:
1. It compiles a **bounds matrix** — upper and lower limits on all pairwise atom distances, derived from bond lengths, angles, and torsional preferences.
2. It samples random distances within those bounds ("distance geometry").
3. It refines the coordinates to satisfy all constraints simultaneously.

**ETKDGv3** (Experimental Torsion Knowledge Distance Geometry, version 3) is the current best method. It augments basic distance geometry with torsional angle preferences mined from the Cambridge Structural Database, producing more realistic conformations out of the box.

**Critical step:** Always call `Chem.AddHs()` before embedding. Hydrogens affect geometry significantly — for example, the difference between a planar sp2 centre and a tetrahedral sp3 centre depends on the hydrogen count.

In [2]:
# Generate a single 3D conformer
mol = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")  # aspirin
mol = Chem.AddHs(mol)  # IMPORTANT: add H before embedding

# ETKDGv3 — best current method
params = AllChem.ETKDGv3()
params.randomSeed = 42
status = AllChem.EmbedMolecule(mol, params)
print(f"Embedding status: {status}")  # 0 = success
print(f"Num conformers: {mol.GetNumConformers()}")

# Get coordinates
conf = mol.GetConformer()
for i in range(min(5, mol.GetNumAtoms())):
    pos = conf.GetAtomPosition(i)
    atom = mol.GetAtomWithIdx(i)
    print(f"  Atom {i} ({atom.GetSymbol():2s}): ({pos.x:.3f}, {pos.y:.3f}, {pos.z:.3f})")

Embedding status: 0
Num conformers: 1
  Atom 0 (C ): (-2.687, -2.329, 0.007)
  Atom 1 (C ): (-1.790, -1.206, -0.305)
  Atom 2 (O ): (-1.658, -0.770, -1.496)
  Atom 3 (O ): (-1.046, -0.564, 0.664)
  Atom 4 (C ): (-0.194, 0.498, 0.387)


## Force Field Optimisation

**Wet-lab refresher — Molecular Mechanics:**

Distance geometry gives you a *reasonable* starting structure, but it is not energy-minimised. Bond lengths and angles may be slightly off, and steric clashes can remain.

Molecular mechanics (MM) force fields approximate the potential energy of a molecule using classical physics terms:

| Term | What it models | Analogy |
|------|---------------|---------|
| Bond stretching | Hooke's law springs between bonded atoms | Stretching a spring on the Dreiding model |
| Angle bending | Resistance to deforming bond angles | Bending the connectors |
| Torsions | Preferences for dihedral angles (gauche, anti, eclipsed) | Rotating around a single bond |
| Van der Waals | Short-range repulsion + London dispersion | Atoms bumping into each other |
| Electrostatics | Coulombic interactions between partial charges | Charge-charge attraction/repulsion |

RDKit provides two force fields:
- **MMFF94** (Merck Molecular Force Field) — well-parameterised for drug-like organic molecules. Your default choice.
- **UFF** (Universal Force Field) — covers the entire periodic table. Use this for organometallics or unusual elements where MMFF94 has no parameters.

Optimisation iteratively adjusts atom positions to find a local energy minimum. This is much faster than quantum mechanics (DFT, MP2, etc.) and perfectly adequate for geometry cleaning and conformer ranking.

In [3]:
# MMFF94 optimization
result = AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
print(f"Optimization converged: {result == 0}")

# Get MMFF energy
ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol))
print(f"MMFF94 energy: {ff.CalcEnergy():.2f} kcal/mol")

Optimization converged: True
MMFF94 energy: 18.91 kcal/mol


In [4]:
# UFF — universal force field, works for more element types
mol2 = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")  # aspirin again
mol2 = Chem.AddHs(mol2)
AllChem.EmbedMolecule(mol2, AllChem.ETKDGv3())
result = AllChem.UFFOptimizeMolecule(mol2)
print(f"UFF optimization converged: {result == 0}")

UFF optimization converged: True


## Conformer Ensembles

**Wet-lab refresher — Conformational Flexibility:**

Molecules are not rigid objects frozen in a single pose. In solution at room temperature, they constantly rotate around single bonds, sampling many conformations. The population of each conformation follows the **Boltzmann distribution** — lower-energy conformers are more populated, but higher-energy ones are still visited.

This matters practically:
- **NMR** measures time-averaged observables (J-couplings, NOEs) over the entire ensemble. A single conformer cannot explain NMR data for a flexible molecule.
- **Bioactive conformation** is often not the global energy minimum — the protein can "pay" for conformational strain through binding interactions.
- **Entropy** contributes to binding free energy. A more flexible molecule loses more conformational entropy upon binding.

Generating a **conformer ensemble** captures this flexibility. We use `EmbedMultipleConfs` to sample diverse 3D arrangements, then optimise each one. The `pruneRmsThresh` parameter discards conformers that are too similar (below a given RMSD threshold), ensuring diversity.

In [5]:
# Generate multiple conformers
mol = Chem.MolFromSmiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O")  # ibuprofen
mol = Chem.AddHs(mol)

params = AllChem.ETKDGv3()
params.randomSeed = 42
params.pruneRmsThresh = 0.5  # RMSD pruning threshold

conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=50, params=params)
print(f"Conformers generated: {len(conf_ids)}")

# Optimize all conformers with MMFF
results = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500)
energies = [r[1] for r in results if r[0] == 0]
print(f"Successfully optimized: {len(energies)}")
print(f"Energy range: {min(energies):.2f} to {max(energies):.2f} kcal/mol")
print(f"Energy spread: {max(energies) - min(energies):.2f} kcal/mol")

Conformers generated: 11
Successfully optimized: 11
Energy range: 23.79 to 25.29 kcal/mol
Energy spread: 1.50 kcal/mol


## Conformer Analysis

With an ensemble in hand, we can now analyse the geometric diversity of the conformers. Key metrics include:

- **Interatomic distances** — directly comparable to NOE-derived distances from NMR.
- **RMSD (Root Mean Square Deviation)** — quantifies how different two conformers are. An RMSD of 0 means identical; >2 A typically means substantially different shapes.
- **RMSD matrix** — pairwise comparison of all conformers, useful for clustering.

In [6]:
# Get the lowest-energy conformer
best_conf_id = int(np.argmin(energies))
conf = mol.GetConformer(best_conf_id)

# Measure distance between two atoms
pos0 = conf.GetAtomPosition(0)
pos1 = conf.GetAtomPosition(1)
dist = pos0.Distance(pos1)
print(f"Distance atom 0-1: {dist:.3f} A")

# RMSD between conformers
rmsd = AllChem.GetConformerRMS(mol, 0, best_conf_id)
print(f"RMSD between conf 0 and best: {rmsd:.3f} A")

# RMSD matrix for first 10 conformers
n = min(10, len(conf_ids))
print(f"\nRMSD matrix (first {n} conformers):")
for i in range(n):
    row = []
    for j in range(n):
        if i == j:
            row.append(0.0)
        else:
            row.append(AllChem.GetConformerRMS(mol, int(conf_ids[i]), int(conf_ids[j])))
    print("  " + "  ".join(f"{v:.2f}" for v in row))

Distance atom 0-1: 1.531 A
RMSD between conf 0 and best: 1.925 A

RMSD matrix (first 10 conformers):
  0.00  2.16  1.87  2.50  2.09  2.20  1.92  2.30  1.64  1.54
  2.16  0.00  2.24  2.14  2.39  2.30  1.85  1.85  2.32  2.29
  1.87  2.24  0.00  1.90  2.06  2.05  2.11  2.16  1.44  1.47
  2.50  2.14  1.90  0.00  2.08  2.34  2.32  1.53  2.24  2.19
  2.09  2.39  2.06  2.08  0.00  1.68  1.88  2.21  1.99  2.28
  2.20  2.30  2.05  2.34  1.68  0.00  1.77  2.22  2.18  1.88
  1.92  1.85  2.11  2.32  1.88  1.77  0.00  2.39  1.86  2.24
  2.30  1.85  2.16  1.53  2.21  2.22  2.39  0.00  1.97  2.29
  1.64  2.32  1.44  2.24  1.99  2.18  1.86  1.97  0.00  1.59
  1.54  2.29  1.47  2.19  2.28  1.88  2.24  2.29  1.59  0.00


## Molecular Alignment

**Wet-lab refresher — Pharmacophore Overlay:**

Molecular overlay (alignment) is a cornerstone of medicinal chemistry SAR analysis. When you align two active compounds and see that their H-bond acceptors, aromatic rings, and hydrophobic groups occupy the same regions of space, you have strong evidence they share a binding mode.

Two alignment methods are available in RDKit:

1. **`AlignMol`** — aligns atoms by index mapping (requires a substructure match or atom map). Fast, but requires you to define the correspondence.
2. **`GetO3A` (Open3D Alignment)** — automatically finds the best overlay based on shape and chemical features (Crippen contributions). No atom mapping needed. This is closer to what you do mentally when you overlay structures in your head.

In [7]:
# Align two molecules
mol1 = Chem.MolFromSmiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O")  # ibuprofen
mol2 = Chem.MolFromSmiles("COC1=CC2=C(C=C1)C=C(C2)C(C)C(=O)O")  # naproxen
mol1 = Chem.AddHs(mol1)
mol2 = Chem.AddHs(mol2)

params = AllChem.ETKDGv3()
params.randomSeed = 42
AllChem.EmbedMolecule(mol1, params)
AllChem.EmbedMolecule(mol2, params)
AllChem.MMFFOptimizeMolecule(mol1)
AllChem.MMFFOptimizeMolecule(mol2)

# Align mol2 onto mol1
rmsd = rdMolAlign.AlignMol(mol2, mol1)
print(f"Alignment RMSD: {rmsd:.3f} A")

# O3A alignment (Open3D Alignment — better for pharmacophore matching)
# Re-embed since AlignMol modified mol2's coordinates
AllChem.EmbedMolecule(mol2, params)
AllChem.MMFFOptimizeMolecule(mol2)

pyO3A = rdMolAlign.GetO3A(mol2, mol1)
score = pyO3A.Align()
print(f"O3A score: {score:.3f}")

RuntimeError: No sub-structure match found between the probe and query mol

## Writing 3D Structures

Once you have generated and optimised conformers, you will want to export them. The most common formats for 3D molecular structures:

- **SDF / MOL** — the standard interchange format for small molecules with 3D coordinates. Stores one or more conformers with full atom/bond information.
- **PDB** — for protein-ligand complexes (standard in structural biology).
- **XYZ** — bare-bones format (element + Cartesian coordinates), common in computational chemistry.

Below we write the ibuprofen conformer ensemble to an SDF-format mol block.

In [8]:
# Write SDF with 3D coordinates (multiple conformers)
mol_for_write = Chem.RemoveHs(mol)  # remove H for cleaner SDF
mol_block = Chem.MolToMolBlock(mol_for_write)
print(mol_block[:300])
print("...")


     RDKit          3D

 15 15  0  0  0  0  0  0  0  0999 V2000
   -4.7599   -0.0664   -0.0237 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.2836    0.2419    0.2381 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.0245    0.2788    1.7465 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.3987   -0.7992   -0
...


## Summary

In this notebook we covered the full pipeline for going from a 2D molecular graph to 3D conformational analysis:

| Step | Function | What it does |
|------|----------|-------------|
| Add hydrogens | `Chem.AddHs()` | Explicit H are needed for correct 3D geometry |
| Embed (single) | `AllChem.EmbedMolecule()` | Distance geometry to generate initial 3D coordinates |
| Embed (ensemble) | `AllChem.EmbedMultipleConfs()` | Sample multiple conformations with RMSD pruning |
| Optimise | `AllChem.MMFFOptimizeMolecule()` | Energy minimisation with MMFF94 force field |
| Optimise (batch) | `AllChem.MMFFOptimizeMoleculeConfs()` | Optimise all conformers at once |
| Analyse distances | `conf.GetAtomPosition()` / `.Distance()` | Measure interatomic distances |
| Analyse RMSD | `AllChem.GetConformerRMS()` | Compare conformer geometries |
| Align molecules | `rdMolAlign.AlignMol()` / `GetO3A()` | Overlay molecules for shape comparison |
| Export | `Chem.MolToMolBlock()` | Write 3D coordinates to SDF/MOL format |

**Key takeaways:**
- Always add explicit hydrogens before embedding -- they determine geometry (sp2 vs. sp3).
- Use **ETKDGv3** for the best conformer generation quality.
- Use **MMFF94** for drug-like molecules; fall back to **UFF** for unusual elements.
- A conformer ensemble (not a single structure) better represents molecular flexibility in solution.
- O3A alignment is the most robust method for overlaying structurally diverse molecules.

**Next up:** Notebook 08 will build on these 3D structures for molecular shape descriptors and pharmacophore feature mapping.